# CocinaAI — EDA e Ingeniería de Features

**Proyecto:** CocinaAI — Recomendador de recetas mexicanas con enfoque Zero Waste  
**Nicho:** Sostenibilidad / Reducción de desperdicio alimentario en México  
**Fuente de datos:** Spoonacular API (cocina mexicana)  

Este notebook cubre:
1. Ingesta de datos desde la API
2. Limpieza y normalización
3. Ingeniería de variables (dificultad, nutrición, Zero Waste, TF-IDF)
4. Análisis exploratorio y visualizaciones
5. Exportación de datasets para modelado

### Imports

In [ ]:
!pip install requests pandas numpy plotly scikit-learn -q

In [ ]:
import json
import requests
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Colab only
from google.colab import userdata

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

### Variables globales

In [ ]:
spoonacular_api = userdata.get('SPOONACULAR_API')
cuisine = 'mexican'
n_recipes = 100  # peticiones al plan free: ajustar si hay límite de cuota

### Funciones

In [ ]:
def get_recipes(cuisine: str, number: int, api_key: str) -> list:
    """
    Obtiene recetas de cocina mexicana desde Spoonacular API.
    Incluye información nutricional e información completa de cada receta.
    """
    url = 'https://api.spoonacular.com/recipes/complexSearch'
    params = {
        'cuisine': cuisine,
        'number': number,
        'addRecipeInformation': True,
        'addRecipeNutrition': True,
        'apiKey': api_key
    }
    response = requests.get(url, params=params)
    return response.json().get('results', [])


In [ ]:
def parse_recipe(recipe: dict) -> dict:
    """
    Extrae y normaliza los campos relevantes de una receta cruda.
    Retorna un diccionario plano listo para convertir a DataFrame.
    """
    nutrition = recipe.get('nutrition', {})
    nutrients = {n['name']: n['amount'] for n in nutrition.get('nutrients', [])}
    ingredients = nutrition.get('ingredients', [])
    ingredient_names = [i['name'].lower().strip() for i in ingredients]

    return {
        'id': recipe.get('id'),
        'titulo': recipe.get('title'),
        'tiempo_minutos': recipe.get('readyInMinutes'),
        'porciones': recipe.get('servings'),
        'precio_por_porcion': recipe.get('pricePerServing'),
        'puntuacion_salud': recipe.get('healthScore'),
        'num_ingredientes': len(ingredients),
        'ingredientes': ingredient_names,
        'calorias': nutrients.get('Calories', np.nan),
        'proteinas_g': nutrients.get('Protein', np.nan),
        'carbohidratos_g': nutrients.get('Carbohydrates', np.nan),
        'grasas_g': nutrients.get('Fat', np.nan),
        'fibra_g': nutrients.get('Fiber', np.nan),
        'sodio_mg': nutrients.get('Sodium', np.nan),
        'vegana': recipe.get('vegan', False),
        'vegetariana': recipe.get('vegetarian', False),
        'sin_gluten': recipe.get('glutenFree', False),
        'url': recipe.get('sourceUrl', ''),
    }


In [ ]:
def clasificar_dificultad(num_ingredientes: int, tiempo_minutos: int) -> str:
    """
    Clasifica la dificultad de una receta en base a ingredientes y tiempo.
    Score compuesto: 0-1 = Fácil, 2-3 = Media, 4+ = Difícil.
    """
    score = 0
    if num_ingredientes > 12: score += 2
    elif num_ingredientes > 7: score += 1
    if tiempo_minutos > 60: score += 2
    elif tiempo_minutos > 30: score += 1
    if score <= 1: return 'Facil'
    elif score <= 3: return 'Media'
    else: return 'Dificil'


In [ ]:
def generate_bar_chart(df, x_col, y_col, title, color='#1D9E75'):
    """Genera gráfica de barras con Plotly."""
    fig = go.Figure(go.Bar(x=df[x_col], y=df[y_col], marker_color=color))
    fig.update_layout(title=title, xaxis_tickangle=-45,
                      template='plotly_white', height=400)
    return fig


## 1. Ingesta de datos

### Fuente de datos

**Spoonacular API** es una base de datos de recetas con más de 380,000 recetas. Para este proyecto filtramos por `cuisine=mexican` para obtener recetas de cocina mexicana.

**Relevancia para Zero Waste:** Spoonacular incluye información de ingredientes con cantidades exactas, lo que permite calcular qué porcentaje de los ingredientes del usuario se aprovechan en cada receta.

In [ ]:
raw_recipes = get_recipes(cuisine=cuisine, number=n_recipes, api_key=spoonacular_api)
print(f'Recetas obtenidas: {len(raw_recipes)}')

In [ ]:
# Estructura de una receta cruda
print('Campos disponibles:')
print(list(raw_recipes[0].keys()))

In [ ]:
recipes_list = [parse_recipe(r) for r in raw_recipes]
recipes_df = pd.DataFrame(recipes_list)
print(f'Shape: {recipes_df.shape}')
recipes_df.head(3)

## 2. Limpieza y calidad de datos

Revisamos valores nulos, tipos de datos y outliers antes de cualquier transformación.

In [ ]:
# Revisión de nulos
nulos = recipes_df.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0])

In [ ]:
# Revisión de tipos
recipes_df.dtypes

In [ ]:
# Estadísticas descriptivas antes de limpieza
recipes_df[['tiempo_minutos','num_ingredientes','calorias',
            'proteinas_g','grasas_g','precio_por_porcion']].describe().round(2)

In [ ]:
# Eliminamos filas con campos críticos nulos
n_antes = len(recipes_df)
recipes_df = recipes_df.dropna(subset=['tiempo_minutos', 'calorias', 'num_ingredientes'])
n_despues = len(recipes_df)
print(f'Filas eliminadas por nulos: {n_antes - n_despues}')
print(f'Recetas limpias: {n_despues}')

In [ ]:
# Eliminamos outliers extremos en tiempo (recetas >480 min son poco realistas para uso diario)
recipes_df = recipes_df[recipes_df['tiempo_minutos'] <= 480]
recipes_df = recipes_df[recipes_df['calorias'] > 0]
print(f'Recetas tras eliminar outliers: {len(recipes_df)}')

In [ ]:
# Resetear índice tras limpieza
recipes_df = recipes_df.reset_index(drop=True)
recipes_df.shape

## 3. Ingeniería de variables

Construimos features nuevas que enriquecen el dataset y son relevantes tanto para el análisis exploratorio como para el modelo de clustering.

### 3.1 Dificultad

**Justificación:** Score compuesto basado en número de ingredientes y tiempo de preparación. Refleja la accesibilidad de la receta para usuarios con distintos niveles de habilidad culinaria.

In [ ]:
recipes_df['dificultad'] = recipes_df.apply(
    lambda row: clasificar_dificultad(row['num_ingredientes'], row['tiempo_minutos']), axis=1
)
recipes_df['dificultad'].value_counts()

### 3.2 Categoría calórica

**Justificación:** Segmentación nutricional en 3 grupos para facilitar recomendaciones a usuarios con distintos objetivos dietéticos.

In [ ]:
def clasificar_calorias(cal):
    if cal < 300: return 'Ligero (<300 cal)'
    elif cal < 600: return 'Moderado (300-600 cal)'
    else: return 'Abundante (>600 cal)'

recipes_df['categoria_calorica'] = recipes_df['calorias'].apply(clasificar_calorias)
recipes_df['categoria_calorica'].value_counts()

### 3.3 Score Zero Waste

**Justificación:** Métrica central del proyecto. Mide qué tan aprovechable es una receta en términos de sostenibilidad. Se construye con 4 sub-features:

- `ingredientes_comunes`: ingredientes de la receta que aparecen en el top 50 más frecuentes (fáciles de tener en despensa)
- `pocos_ingredientes`: recetas con ≤7 ingredientes generan menos desperdicio
- `es_economica`: precio por porción menor al percentil 40
- `score_zero_waste`: promedio ponderado de las 3 variables anteriores (0 a 1)

In [ ]:
# Calculamos ingredientes frecuentes (top 50 en todo el dataset)
all_ingredients = [ing for sublist in recipes_df['ingredientes'] for ing in sublist]
ingredient_counts = Counter(all_ingredients)
top_50_ingredientes = set([ing for ing, _ in ingredient_counts.most_common(50)])

# Proporción de ingredientes de la receta que están en el top 50 (fácil de tener en casa)
recipes_df['prop_ingredientes_comunes'] = recipes_df['ingredientes'].apply(
    lambda ings: len(set(ings) & top_50_ingredientes) / len(ings) if len(ings) > 0 else 0
).round(3)

# Recetas con pocos ingredientes (menos desperdicio potencial)
recipes_df['pocos_ingredientes'] = (recipes_df['num_ingredientes'] <= 7).astype(int)

# Recetas económicas (por debajo del percentil 40 en precio)
precio_p40 = recipes_df['precio_por_porcion'].quantile(0.40)
recipes_df['es_economica'] = (recipes_df['precio_por_porcion'] <= precio_p40).astype(int)

# Score Zero Waste compuesto (pesos: 50% ingredientes comunes, 30% pocos ingredientes, 20% economía)
recipes_df['score_zero_waste'] = (
    0.50 * recipes_df['prop_ingredientes_comunes'] +
    0.30 * recipes_df['pocos_ingredientes'] +
    0.20 * recipes_df['es_economica']
).round(3)

print('Score Zero Waste — estadísticas:')
recipes_df['score_zero_waste'].describe().round(3)

In [ ]:
# Top 10 recetas con mayor score Zero Waste
recipes_df[['titulo','num_ingredientes','prop_ingredientes_comunes',
            'score_zero_waste','dificultad']].sort_values(
    'score_zero_waste', ascending=False).head(10)

### 3.4 Ratio proteína/carbohidratos

**Justificación:** Feature nutricional relevante para usuarios con objetivos específicos (alto en proteína, bajo en carbs). Complementa la categoría calórica.

In [ ]:
recipes_df['ratio_proteina_carbs'] = (
    recipes_df['proteinas_g'] / recipes_df['carbohidratos_g'].replace(0, np.nan)
).round(3)
recipes_df['ratio_proteina_carbs'].describe().round(3)

### 3.5 Normalización (MinMaxScaler)

**Justificación:** Las columnas numéricas tienen escalas muy distintas (calorías en cientos, tiempo en minutos, precio en centavos de dólar). Normalizamos al rango [0,1] para que el algoritmo de clustering no sea dominado por variables de mayor magnitud.

In [ ]:
cols_a_normalizar = [
    'tiempo_minutos', 'num_ingredientes', 'calorias',
    'proteinas_g', 'carbohidratos_g', 'grasas_g', 'fibra_g',
    'precio_por_porcion', 'puntuacion_salud', 'score_zero_waste'
]

scaler = MinMaxScaler()
normalized_values = scaler.fit_transform(recipes_df[cols_a_normalizar].fillna(0))
normalized_df = pd.DataFrame(
    normalized_values,
    columns=[f'{c}_norm' for c in cols_a_normalizar],
    index=recipes_df.index
)

# Añadimos al dataframe principal
recipes_df = pd.concat([recipes_df, normalized_df], axis=1)
print('Columnas normalizadas agregadas.')
normalized_df.describe().round(3)

### 3.6 Vectorización semántica de ingredientes (TF-IDF)

**Justificación:** Para el modelo de clustering necesitamos representar cada receta como un vector numérico basado en sus ingredientes. TF-IDF (Term Frequency-Inverse Document Frequency) asigna mayor peso a ingredientes que son frecuentes en una receta pero poco comunes en el resto del dataset — capturando la 'firma' de cada platillo.

Cada receta se convierte en un 'documento' donde los 'términos' son sus ingredientes.

In [ ]:
# Convertimos la lista de ingredientes a string para TF-IDF
recipes_df['ingredientes_texto'] = recipes_df['ingredientes'].apply(
    lambda ings: ' '.join([i.replace(' ', '_') for i in ings])
)

# Aplicamos TF-IDF — máximo 200 features (ingredientes únicos más relevantes)
tfidf = TfidfVectorizer(
    max_features=200,
    ngram_range=(1, 2),  # incluye pares de ingredientes (ej: 'chile_serrano')
    min_df=2             # ignora ingredientes que aparecen en menos de 2 recetas
)

tfidf_matrix = tfidf.fit_transform(recipes_df['ingredientes_texto'])
print(f'Matriz TF-IDF: {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]} recetas x {tfidf_matrix.shape[1]} features de ingredientes')

In [ ]:
# Top 20 ingredientes con mayor peso promedio en el corpus
feature_names = tfidf.get_feature_names_out()
avg_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_tfidf_df = pd.DataFrame({'ingrediente': feature_names, 'peso_tfidf': avg_tfidf})
top_tfidf_df = top_tfidf_df.sort_values('peso_tfidf', ascending=False).head(20)
top_tfidf_df

## 4. Análisis exploratorio

### 4.1 Distribuciones generales

In [ ]:
recipes_df[['tiempo_minutos','num_ingredientes','calorias',
            'proteinas_g','score_zero_waste','precio_por_porcion']].describe().round(2)

#### Distribución de tiempos de preparación

In [ ]:
fig = px.histogram(
    recipes_df, x='tiempo_minutos', nbins=20,
    title='Distribución de tiempos de preparación (minutos)',
    color_discrete_sequence=['#534AB7'], template='plotly_white'
)
fig.show()

#### Top 20 ingredientes más frecuentes

In [ ]:
top_ingredients_df = pd.DataFrame(
    ingredient_counts.most_common(20), columns=['ingrediente', 'frecuencia']
)
fig = generate_bar_chart(
    top_ingredients_df, 'ingrediente', 'frecuencia',
    'Top 20 ingredientes más frecuentes en cocina mexicana', color='#1D9E75'
)
fig.show()

### 4.2 Análisis Zero Waste

In [ ]:
# Distribución del Score Zero Waste
fig = px.histogram(
    recipes_df, x='score_zero_waste', nbins=15,
    title='Distribución del Score Zero Waste por receta',
    color_discrete_sequence=['#1D9E75'], template='plotly_white'
)
fig.add_vline(x=recipes_df['score_zero_waste'].mean(), line_dash='dash',
              line_color='#D85A30', annotation_text='Promedio')
fig.show()

In [ ]:
# Score Zero Waste por dificultad
zw_dificultad = recipes_df.groupby('dificultad')['score_zero_waste'].mean().round(3).reset_index()
fig = generate_bar_chart(
    zw_dificultad, 'dificultad', 'score_zero_waste',
    'Score Zero Waste promedio por nivel de dificultad', color='#1D9E75'
)
fig.show()

print('Hallazgo: las recetas más simples tienden a tener mayor score Zero Waste')
print('porque usan menos ingredientes y más ingredientes comunes de despensa.')

In [ ]:
# Relación entre número de ingredientes y score Zero Waste
fig = px.scatter(
    recipes_df, x='num_ingredientes', y='score_zero_waste',
    color='dificultad', hover_name='titulo', size='calorias',
    title='Ingredientes vs Score Zero Waste',
    template='plotly_white',
    color_discrete_map={'Facil': '#1D9E75', 'Media': '#EF9F27', 'Dificil': '#D85A30'}
)
fig.show()

### 4.3 Análisis nutricional

In [ ]:
# Calorías por dificultad
cal_dif = recipes_df.groupby('dificultad')['calorias'].mean().round(1).reset_index()
fig = generate_bar_chart(cal_dif, 'dificultad', 'calorias',
    'Calorías promedio por nivel de dificultad', color='#D85A30')
fig.show()

In [ ]:
# Tipos de dieta
dieta_counts = pd.DataFrame({
    'tipo': ['Vegana', 'Vegetariana', 'Sin gluten'],
    'cantidad': [recipes_df['vegana'].sum(),
                 recipes_df['vegetariana'].sum(),
                 recipes_df['sin_gluten'].sum()]
})
fig = generate_bar_chart(dieta_counts, 'tipo', 'cantidad',
    'Recetas por tipo de dieta', color='#378ADD')
fig.show()

### 4.4 Peso TF-IDF de ingredientes

In [ ]:
fig = generate_bar_chart(
    top_tfidf_df, 'ingrediente', 'peso_tfidf',
    'Top 20 ingredientes por peso TF-IDF (identidad de la cocina mexicana)',
    color='#534AB7'
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print('Estos ingredientes son los más distintivos de la cocina mexicana en el dataset.')
print('Un peso alto indica que el ingrediente aparece frecuentemente en ciertas recetas')
print('pero no en todas, lo que lo hace informativo para diferenciar platillos.')

### 4.5 Tablas resumen

In [ ]:
# Tabla resumen por dificultad
dificultad_table = recipes_df.groupby('dificultad').agg(
    num_recetas=('id','count'),
    tiempo_promedio=('tiempo_minutos','mean'),
    calorias_promedio=('calorias','mean'),
    ingredientes_promedio=('num_ingredientes','mean'),
    score_zw_promedio=('score_zero_waste','mean')
).round(2).reset_index()
dificultad_table

In [ ]:
# Top 10 recetas más Zero Waste y rápidas
recipes_df[['titulo','tiempo_minutos','num_ingredientes',
            'score_zero_waste','dificultad']].sort_values(
    ['score_zero_waste','tiempo_minutos'], ascending=[False, True]).head(10)

## 5. Exportación

Guardamos tres datasets:
- `recipes_df.csv` — dataset principal con todas las features
- `tfidf_matrix.npz` — matriz TF-IDF esparsa para clustering
- `tfidf_feature_names.json` — nombres de features del TF-IDF

In [ ]:
# Export del dataframe principal
export_df = recipes_df.copy()
export_df['ingredientes'] = export_df['ingredientes'].apply(lambda x: ', '.join(x))
# Descartamos columnas intermedias que no se necesitan en los siguientes notebooks
export_df = export_df.drop(columns=['ingredientes_texto'], errors='ignore')
export_df.to_csv('recipes_df.csv', index=False)
print(f'recipes_df.csv exportado — {export_df.shape[0]} recetas, {export_df.shape[1]} columnas')

In [ ]:
# Export de la matriz TF-IDF (formato esparso para eficiencia)
scipy.sparse.save_npz('tfidf_matrix.npz', tfidf_matrix)
with open('tfidf_feature_names.json', 'w') as f:
    json.dump(list(tfidf.get_feature_names_out()), f)
print('tfidf_matrix.npz exportado')
print('tfidf_feature_names.json exportado')

In [ ]:
# Tabla resumen de features generadas
features_nuevas = [
    ('dificultad', 'Categórica', 'Score compuesto: ingredientes + tiempo'),
    ('categoria_calorica', 'Categórica', 'Segmentación nutricional en 3 grupos'),
    ('prop_ingredientes_comunes', 'Numérica [0-1]', 'Proporción de ingredientes en top 50'),
    ('pocos_ingredientes', 'Binaria', '1 si tiene 7 o menos ingredientes'),
    ('es_economica', 'Binaria', '1 si precio < percentil 40'),
    ('score_zero_waste', 'Numérica [0-1]', 'Score central del proyecto — aprovechamiento'),
    ('ratio_proteina_carbs', 'Numérica', 'Ratio proteína / carbohidratos'),
    ('*_norm', 'Numérica [0-1]', 'Versiones normalizadas (MinMaxScaler) para clustering'),
    ('ingredientes_texto', 'Texto', 'Ingredientes unidos para vectorización TF-IDF'),
]
features_df = pd.DataFrame(features_nuevas, columns=['Feature','Tipo','Descripción'])
features_df